# CM3 - Traitement BigData avec Spark
> **Objectifs**
>
> À la fin de ce chapitre, vous saurez :
> - expliquer pourquoi pandas peut échouer sur un grand volume de données, et ce que Spark change concrètement ;
> - comprendre l'architecture interne de Spark et Comment il distribue le calcul sur un cluster de machines.
> - décrire ce qu'est un RDD, ce qu'est une partition, et distinguer une transformation d'une action ;
> - lire un plan d'exécution simple (DAG) et expliquer pourquoi Spark attend une action pour calculer quoi que ce soit ;
> - manipuler un DataFrame Spark SQL : sélection, colonnes calculées, agrégations, fenêtres, jointures ;
> - situer Spark dans une architecture managée (Google Cloud Dataproc) sans avoir à l'opérer soi-même.


## 1. Le problème du passage à l'échelle
### 1.1 pandas : tout sur une seule machine

pandas charge tout en mémoire, sur un seul processus, une seule machine. Tant que ça tient dans la RAM, c'est rapide : `read_csv`, `groupby`, `merge` tournent en quelques secondes.

Le problème, c'est ce qui se passe quand ça ne tient plus. pandas ne ralentit pas progressivement — il plante. `MemoryError`. Et attendre plus longtemps n'y changera rien : ce n'est pas un problème de vitesse, c'est un problème d'espace.

Même calcul (un max), sur des fichiers Parquet de plus en plus gros, pandas vs PySpark :

![pandas-vs-pyspark](/media/CM3/pandas-vs-pyspark.png)

pandas : erreur out-of-memory vers 36 Go.
PySpark : continue jusqu'à 70 Go, avec un temps d'exécution qui augmente doucement.

### 1.2 La solution de Spark : distribuer

Spark ne cherche pas une machine plus puissante (scale-up). Il découpe la table et la répartit sur plusieurs machines — les executors — chacune ne portant qu'un morceau (scale-out).